# RAG Introduction with Text File Upload

## Complete Student-Friendly Notebook with Runnables

### Topic
**RAG: Retrieval-Augmented Generation**

### Simple Meaning

RAG means:

```text
Search from document first → Give the searched content to AI → AI gives answer
```

This notebook shows how to upload a `.txt` file from your computer into Google Colab and build a simple RAG Q&A app.

This version also includes **LangChain Runnables**.


## Learning Objectives

By the end of this notebook, students will be able to:

1. Upload a text file into Google Colab.
2. Load the uploaded text file.
3. Split the document into small chunks.
4. Convert text into embeddings.
5. Store embeddings in Chroma vector database.
6. Search relevant content from the document.
7. Create a RAG chain using LangChain Runnables.
8. Ask questions using the RAG chain.
9. Create a simple Gradio Q&A app.


## Step 1: Install Required Libraries

Run this cell first.

It installs LangChain, Groq, Chroma, sentence-transformers, and Gradio.


In [1]:
!pip install -q -U langchain langchain-core langchain-groq langchain-community langchain-chroma langchain-huggingface langchain-text-splitters
!pip install -q chromadb sentence-transformers gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.3/573.3 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.

## Step 2: Load Groq API Key

In Google Colab, save your Groq API key in:

```text
Secrets → groq
```

Make sure notebook access is ON.

Your secret name should be:

```text
groq
```


In [2]:
from google.colab import userdata
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except:
    GROQ_API_KEY = "your-key"
print("Groq API key loaded successfully.")

Groq API key loaded successfully.


## Step 3: Upload Text File from Computer

Upload your file:

```text
python_guide_for_rag.txt
```

You can also upload any `.txt` file.

After upload, the notebook will automatically use the uploaded file.


In [3]:
from google.colab import files

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

print("Uploaded file:", file_name)

Saving python_guide_for_rag.txt to python_guide_for_rag.txt
Uploaded file: python_guide_for_rag.txt


## Step 4: Preview Uploaded Text File

This cell shows the first part of the uploaded file.


In [4]:
with open(file_name, "r", encoding="utf-8") as file:
    text_content = file.read()

print("File loaded successfully.")
print("Total characters:", len(text_content))
print("\nPreview:")
print(text_content[:1000])

File loaded successfully.
Total characters: 1967

Preview:
Python Programming Guide for Beginners

1. Variables
Variables are used to store data in Python.
A variable can store text, numbers, or other values.

Example:
name = "John"
age = 20

Here, name stores text and age stores a number.

2. Data Types
Python has different data types.

Common data types:
- int: stores whole numbers
- float: stores decimal numbers
- str: stores text
- bool: stores True or False values

Example:
student_name = "Alex"
mark = 85
percentage = 89.5
is_passed = True

3. Lists
A list is used to store multiple values in one variable.

Example:
fruits = ["apple", "banana", "cherry"]

We can access list items using index numbers.

Example:
print(fruits[0])

Output:
apple

4. Functions
Functions are reusable blocks of code.
A function helps us avoid writing the same code again and again.

Example:
def greet(name):
    return "Hello " + name

print(greet("John"))

Output:
Hello John

5. Loops
Loops are used to re

## Step 5: Load the Text File using LangChain

We use `TextLoader` to load the uploaded text file.


In [5]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(file_name, encoding="utf-8")

documents = loader.load()

print("Number of documents loaded:", len(documents))
print("\nDocument preview:")
print(documents[0].page_content[:1000])

/tmp/ipykernel_2320/3395100581.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Number of documents loaded: 1

Document preview:
Python Programming Guide for Beginners

1. Variables
Variables are used to store data in Python.
A variable can store text, numbers, or other values.

Example:
name = "John"
age = 20

Here, name stores text and age stores a number.

2. Data Types
Python has different data types.

Common data types:
- int: stores whole numbers
- float: stores decimal numbers
- str: stores text
- bool: stores True or False values

Example:
student_name = "Alex"
mark = 85
percentage = 89.5
is_passed = True

3. Lists
A list is used to store multiple values in one variable.

Example:
fruits = ["apple", "banana", "cherry"]

We can access list items using index numbers.

Example:
print(fruits[0])

Output:
apple

4. Functions
Functions are reusable blocks of code.
A function helps us avoid writing the same code again and again.

Example:
def greet(name):
    return "Hello " + name

print(greet("John"))

Output:
Hello John

5. Loops
Loops are used to repeat code.

## Step 6: Split Document into Chunks

Large documents are split into small parts called **chunks**.

Why?

Because RAG searches only the most relevant chunks instead of reading the full document every time.


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Number of chunks created:", len(chunks))

for i, chunk in enumerate(chunks[:5]):
    print("\nChunk", i + 1)
    print(chunk.page_content)
    print("-" * 50)

Number of chunks created: 9

Chunk 1
Python Programming Guide for Beginners

1. Variables
Variables are used to store data in Python.
A variable can store text, numbers, or other values.

Example:
name = "John"
age = 20

Here, name stores text and age stores a number.

2. Data Types
Python has different data types.
--------------------------------------------------

Chunk 2
2. Data Types
Python has different data types.

Common data types:
- int: stores whole numbers
- float: stores decimal numbers
- str: stores text
- bool: stores True or False values

Example:
student_name = "Alex"
mark = 85
percentage = 89.5
is_passed = True
--------------------------------------------------

Chunk 3
3. Lists
A list is used to store multiple values in one variable.

Example:
fruits = ["apple", "banana", "cherry"]

We can access list items using index numbers.

Example:
print(fruits[0])

Output:
apple
--------------------------------------------------

Chunk 4
Example:
print(fruits[0])

Output:
apple

## Step 7: Create Embeddings

### What are embeddings?

Embeddings convert text into numbers.

The computer cannot understand text directly, so embeddings help the system search similar meanings.


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded successfully.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings model loaded successfully.


## Step 8: Create Chroma Vector Store

The vector store saves document chunks as embeddings.

It helps us search relevant information from the document.


In [10]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="student_rag_collection"
)

print("Vector store created successfully.")

Vector store created successfully.


## Step 9: Create RAG Chain using Runnables

This step uses LangChain Runnable style.

### Simple meaning

```text
Question
↓
Retriever searches document
↓
Relevant chunks become context
↓
Prompt is created
↓
LLM gives answer
↓
Output parser returns text
```

We use:

| Component | Use |
|---|---|
| `retriever` | Searches relevant document chunks |
| `RunnablePassthrough()` | Passes the user question forward |
| `ChatPromptTemplate` | Creates the final prompt |
| `llm` | Generates the answer |
| `StrOutputParser()` | Converts output into normal text |


In [11]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Create LLM
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model="openai/gpt-oss-20b"
)

# Create retriever from vector store
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

# Function to join retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create RAG prompt
rag_prompt = ChatPromptTemplate.from_template(
    '''
Answer the question using only the context below.
If the answer is not available in the context, say "I don't know."

Context:
{context}

Question:
{question}

Answer:
'''
)

# Create Runnable RAG chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("Runnable RAG chain created successfully.")

Runnable RAG chain created successfully.


## Step 10: Test the Runnable RAG Chain

Now we ask a question.

The chain will automatically:

```text
Search → Create Prompt → Ask LLM → Return Answer
```


In [12]:
question = "How do I create a variable in Python?"

answer = rag_chain.invoke(question)

print("Question:", question)
print("\nAnswer:")
print(answer)

Question: How do I create a variable in Python?

Answer:
Use an assignment statement.  
```python
name = "John"   # creates a variable called name that holds a string
age = 20        # creates a variable called age that holds a number
```
The left side is the variable name, and the right side is the value you want to store.


## Step 11: View Retrieved Chunks

This cell shows what content was retrieved from the document.

This helps students understand how RAG searches before answering.


In [13]:
question = "What are functions?"

retrieved_docs = retriever.invoke(question)

print("Question:", question)
print("\nRetrieved chunks:")

for i, doc in enumerate(retrieved_docs):
    print("\nChunk", i + 1)
    print(doc.page_content)
    print("-" * 60)

Question: What are functions?

Retrieved chunks:

Chunk 1
Example:
print(fruits[0])

Output:
apple

4. Functions
Functions are reusable blocks of code.
A function helps us avoid writing the same code again and again.

Example:
def greet(name):
    return "Hello " + name

print(greet("John"))

Output:
Hello John

5. Loops
Loops are used to repeat code.
------------------------------------------------------------

Chunk 2
Students should practice variables, data types, lists, functions, loops, conditions, and dictionaries to build a strong Python foundation.
------------------------------------------------------------

Chunk 3
10. Summary
Python is a beginner-friendly programming language.
It is used in web development, data science, machine learning, automation, and artificial intelligence.
------------------------------------------------------------


## Step 12: Ask Multiple Questions using Runnable Chain

You can change the questions based on your uploaded text file.


In [14]:
questions = [
    "What are variables?",
    "What is a function?",
    "What is a list?",
    "How do java work?"
]

for q in questions:
    print("\nQuestion:", q)
    print("Answer:", rag_chain.invoke(q))
    print("-" * 70)


Question: What are variables?
Answer: Variables are used to store data in Python.
----------------------------------------------------------------------

Question: What is a function?
Answer: A function is a reusable block of code that performs a specific task, allowing you to avoid writing the same code multiple times.
----------------------------------------------------------------------

Question: What is a list?
Answer: A list is used to store multiple values in one variable.
----------------------------------------------------------------------

Question: How do java work?
Answer: I don't know.
----------------------------------------------------------------------


## Step 13: Simple Function Wrapper

For Gradio, we create a simple function that calls the runnable chain.

This keeps the Gradio code easy for students.


In [15]:
def ask_rag(question):
    if question.strip() == "":
        return "Please enter a question."

    return rag_chain.invoke(question)

print(ask_rag("What are variables?"))

Variables are used to store data in Python.


# Mini Project: RAG Q&A App using Gradio

This app lets students ask questions from the uploaded text file.

The answer box is multiline using:

```python
lines=10
max_lines=25
```


In [17]:

import gradio as gr

def answer_question(question):
    return ask_rag(question)

app = gr.Interface(
    fn=answer_question,

    inputs=gr.Textbox(
        label="Your Question",
        lines=3,
        placeholder="Ask a question from your uploaded document..."
    ),

    outputs=gr.Textbox(
        label="Answer",
        lines=10,
        max_lines=25
    ),

    title="RAG Q&A App using Runnables",
    description="Ask questions based on your uploaded text document.",

    examples=[
        "What are variables?",
        "How do I use loops?",
        "Explain functions",
        "What is a list?"
    ]
)

app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://36c6e22601293b7238.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Runnable RAG Flow

```text
User Question
↓
RunnablePassthrough sends question forward
↓
Retriever searches vector store
↓
format_docs combines retrieved chunks
↓
Prompt is created
↓
LLM generates answer
↓
StrOutputParser returns text
↓
Gradio displays answer
```


## Why Learn Runnables?

The simple function method is easy for beginners.

Runnable method is useful because it helps us build professional AI workflows.

| Simple Function | Runnable Chain |
|---|---|
| Easy to understand | Better for real applications |
| Good for beginners | Good for production-style AI apps |
| Manual steps | Connected pipeline |
| Useful for teaching basics | Useful for LangChain projects |

### Main idea

```text
Runnables connect multiple AI steps into one clean pipeline.
```


## Student Exercises

### Exercise 1
Upload a different `.txt` file and test the RAG system.

### Exercise 2
Change `k=3` to `k=1` and compare the output.

### Exercise 3
Ask five questions from your uploaded document.

### Exercise 4
Change the prompt instruction and observe the answer.

### Exercise 5
Change the Gradio app title and examples.

---

## Final Summary

In this notebook, students learned:

1. How to upload a text file into Colab.
2. How to load a document.
3. How to split documents into chunks.
4. How to create embeddings.
5. How to use Chroma vector store.
6. How to create a retriever.
7. How to create a RAG chain using Runnables.
8. How to ask questions using RAG.
9. How to build a Gradio Q&A app.

### Key Point

RAG helps AI answer using your own document.

Runnables help connect each RAG step into one pipeline.
